# Задание 1. Закон Ципфа

Для каждого из четырёх частотных словарей проверяется формула

$$f \cdot r = c,$$

где $f$ — частота слова, $r$ — его ранг в частотном списке, $c$ — коэффициент Ципфа.

Коэффициент $c$ оценивается как медиана произведения $f \cdot r$ в диапазоне рангов 100–5000. Затем строятся графики зависимости частоты от ранга и произведения $f \cdot r$ от ранга.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'freq_ru.json').exists():
    ROOT = ROOT.parent
BASE = ROOT / 'rgr2'
DATA = BASE / 'data'
FIG = BASE / 'figures'
DATA.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'legend.fontsize': 8,
    'axes.grid': True,
    'grid.alpha': 0.25,
})

DICTIONARIES = [
    ('ru_raw',   'Русский, без лемматизации',  'freq_ru.json'),
    ('ru_lemma', 'Русский, с лемматизацией',   'freq_ru_lemma.json'),
    ('en_raw',   'Английский, без лемматизации', 'freq_en.json'),
    ('en_lemma', 'Английский, с лемматизацией',  'freq_en_lemma.json'),
]

ZIPF_RANK_MIN = 100
ZIPF_RANK_MAX = 5000


def load_dictionary(filename):
    with open(ROOT / filename, encoding='utf-8') as f:
        package = json.load(f)
    items = sorted(package['dictionary'].items(), key=lambda p: (-p[1], p[0]))
    df = pd.DataFrame(items, columns=['word', 'freq'])
    df['rank'] = np.arange(1, len(df) + 1)
    df['rel_freq'] = df['freq'] / package['total_tokens']
    df['freq_x_rank'] = df['freq'] * df['rank']
    return package, df

datasets = {}
for key, label, filename in DICTIONARIES:
    package, df = load_dictionary(filename)
    datasets[key] = {'package': package, 'df': df, 'label': label, 'file': filename}
    print(f"{label}: токенов {package['total_tokens']:,}, уникальных слов {len(df):,}")


Русский, без лемматизации: токенов 4,982,552, уникальных слов 314,763


Русский, с лемматизацией: токенов 5,000,251, уникальных слов 181,819


Английский, без лемматизации: токенов 5,002,249, уникальных слов 142,303


Английский, с лемматизацией: токенов 5,002,249, уникальных слов 124,040


In [2]:
rows = []
for key, label, filename in DICTIONARIES:
    package = datasets[key]['package']
    df = datasets[key]['df']
    sub = df[(df['rank'] >= ZIPF_RANK_MIN) & (df['rank'] <= ZIPF_RANK_MAX)]
    c = float(np.median(sub['freq_x_rank']))

    rows.append({
        'key': key,
        'dictionary': label,
        'file': filename,
        'total_tokens': int(package['total_tokens']),
        'unique_words': int(len(df)),
        'rank_min': ZIPF_RANK_MIN,
        'rank_max': ZIPF_RANK_MAX,
        'c_median': c,
    })

summary = pd.DataFrame(rows)
summary.to_csv(DATA / 'zipf_summary.csv', index=False, float_format='%.8f')

display(summary[['dictionary', 'total_tokens', 'unique_words', 'c_median']]
        .rename(columns={
            'dictionary': 'Словарь',
            'total_tokens': 'Токенов',
            'unique_words': 'Уникальных слов',
            'c_median': 'c',
        }).round(0))


,Словарь,Токенов,Уникальных слов,c
0,"Русский, без лемматизации",4982552,314763,591965.0
1,"Русский, с лемматизацией",5000251,181819,656775.0
2,"Английский, без лемматизации",5002249,142303,581872.0
3,"Английский, с лемматизацией",5002249,124040,567200.0


## Построение графиков

На верхних панелях показаны эмпирические точки в координатах «ранг — частота» (обе оси логарифмические) и прямая $f=c/r$. На нижних панелях — произведение $f \cdot r$; горизонтальная линия соответствует коэффициенту $c$.


In [3]:
def make_zipf_figure(keys, title, out_path):
    fig, axes = plt.subplots(2, 2, figsize=(11.2, 8.0))
    colors = ['#1f77b4', '#d62728']

    for col, key in enumerate(keys):
        item = datasets[key]
        df = item['df']
        c = float(summary.set_index('key').loc[key, 'c_median'])

        ax = axes[0, col]
        sub = df.iloc[::max(1, len(df) // 7000)]
        ax.scatter(sub['rank'], sub['freq'], s=4, alpha=0.30, color=colors[col],
                   linewidths=0, label='Эмпирические данные')
        r_grid = np.logspace(0, np.log10(df['rank'].max()), 400)
        ax.plot(r_grid, c / r_grid, color='black', linestyle='--', linewidth=1.4,
                label=f'$f=c/r$, c = {c:,.0f}'.replace(',', ' '))
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_title(item['label'])
        ax.set_xlabel('Ранг слова r')
        ax.set_ylabel('Частота f')
        ax.legend(loc='lower left')

        ax = axes[1, col]
        ax.scatter(sub['rank'], sub['freq_x_rank'], s=4, alpha=0.30,
                   color=colors[col], linewidths=0)
        ax.axhline(c, color='black', linestyle='--', linewidth=1.4,
                   label=f'c = {c:,.0f}'.replace(',', ' '))
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_title(f"Произведение f·r: {item['label']}")
        ax.set_xlabel('Ранг слова r')
        ax.set_ylabel('f · r')
        ax.legend(loc='lower left')

    fig.suptitle(title, fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)
    print('Сохранено:', out_path)


make_zipf_figure(['ru_raw', 'ru_lemma'], 'Закон Ципфа для русского языка', FIG / 'fig_zipf_ru.png')
make_zipf_figure(['en_raw', 'en_lemma'], 'Закон Ципфа для английского языка', FIG / 'fig_zipf_en.png')


Сохранено: /home/dd/nlp_uni/rgr2/figures/fig_zipf_ru.png


Сохранено: /home/dd/nlp_uni/rgr2/figures/fig_zipf_en.png


In [4]:
lines = ['### Вывод по заданию 1', '']
for _, row in summary.iterrows():
    c = f"{row['c_median']:,.0f}".replace(',', ' ')
    lines.append(f"- **{row['dictionary']}**: c = {c}.")

lines += [
    '',
    'Во всех четырёх словарях произведение $f \cdot r$ в средней части частотного списка '
    'остаётся приблизительно постоянным. Это подтверждает выполнение закона Ципфа '
    'как приближённой закономерности. На краях списка наблюдаются отклонения: '
    'у самых частых слов произведение повышено, у редких — понижено.'
]
display(Markdown('\n'.join(lines)))


<>:8: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:8: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
/tmp/ipykernel_93636/2154710655.py:8: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
  'Во всех четырёх словарях произведение $f \cdot r$ в средней части частотного списка '


### Вывод по заданию 1

- **Русский, без лемматизации**: c = 591 965.
- **Русский, с лемматизацией**: c = 656 775.
- **Английский, без лемматизации**: c = 581 872.
- **Английский, с лемматизацией**: c = 567 200.

Во всех четырёх словарях произведение $f \cdot r$ в средней части частотного списка остаётся приблизительно постоянным. Это подтверждает выполнение закона Ципфа как приближённой закономерности. На краях списка наблюдаются отклонения: у самых частых слов произведение повышено, у редких — понижено.